# 06b: Cross-Ablation — Configs C & D (ReLU6 variants)

**Part 2 of 2** for the activation × attention cross-ablation.
Runs Config C (ReLU6+SE) and Config D (ReLU6+ECA), 3 seeds each = 6 runs.

**Estimated time:** ~31h on T4. If on Kaggle (12h limit), run 2 seeds per session.

In [ ]:
import os, sys, socket
try:
    socket.create_connection(("github.com", 80), timeout=5)
except:
    print("\u274c ERROR: No Internet connection.")
    sys.exit(1)

!rm -rf tinyYOLO && git clone https://github.com/ShMazumder/tinyYOLO.git
%cd tinyYOLO
!pip install -e . -q
!pip install tqdm timm -q
!python3 -c "from ultralytics.data.utils import check_det_dataset; check_det_dataset('VOC.yaml')"

In [ ]:
from tinyYOLO.models import build_model
import torch
x = torch.randn(1, 3, 416, 416)
for v, a, l in [('quantized','se','C: ReLU6+SE'), ('quantized','default','D: ReLU6+ECA')]:
    m, i = build_model(task='det', variant=v, nc=20, attention=a)
    _ = m(x)
    print(f'  \u2713 {l}: {i["total_params_M"]}M, attn3={m.backbone.attn3.__class__.__name__}')
print('Verified!')

In [ ]:
# Config C: ReLU6 + SE/Spatial (cross — isolates ReLU6 effect with SE)
SEEDS = [42, 123, 256]
for s in SEEDS:
    !python3 scripts/train.py --task det --variant quantized --attention se \
        --data voc.yaml --imgsz 416 --epochs 300 --batch 128 --seed {s} \
        --warmup 3 --pretrained --compile --name xabl-C-relu6-se-seed{s}

In [ ]:
# Config D: ReLU6 + ECA (= TinyYOLO-q baseline, should reproduce ~41.2%)
SEEDS = [42, 123, 256]
for s in SEEDS:
    !python3 scripts/train.py --task det --variant quantized --attention default \
        --data voc.yaml --imgsz 416 --epochs 300 --batch 128 --seed {s} \
        --warmup 3 --pretrained --compile --name xabl-D-relu6-eca-seed{s}

## Results Analysis
Run this after BOTH 06a and 06b are complete. Collect all 12 results.

In [ ]:
import json, glob
from pathlib import Path
import numpy as np

results = {}
for config_id, pattern in [
    ('A (SiLU+SE)',  'xabl-A-silu-se-*'),
    ('B (SiLU+ECA)', 'xabl-B-silu-eca-*'),
    ('C (ReLU6+SE)', 'xabl-C-relu6-se-*'),
    ('D (ReLU6+ECA)','xabl-D-relu6-eca-*'),
]:
    maps = []
    for d in sorted(glob.glob(f'experiments/results/{pattern}')):
        summary = Path(d) / 'summary.json'
        if summary.exists():
            with open(summary) as f:
                data = json.load(f)
            maps.append(data.get('best_map50', 0) * 100)
    results[config_id] = maps

print('=' * 70)
print('CROSS-ABLATION RESULTS: Activation x Attention')
print('=' * 70)
print(f'{"Config":<20} {"Runs":>5} {"mAP@50 (%)":>12} {"Std":>8}')
print('-' * 70)
for cfg, maps in results.items():
    if maps:
        print(f'{cfg:<20} {len(maps):>5} {np.mean(maps):>12.1f} {np.std(maps):>8.2f}')
    else:
        print(f'{cfg:<20} {"N/A":>5} {"---":>12} {"---":>8}')
print('=' * 70)

if all(len(v) >= 2 for v in results.values()):
    a, b, c, d = [np.mean(results[k]) for k in results]
    eca_effect = ((b - a) + (d - c)) / 2
    relu6_effect = ((c - a) + (d - b)) / 2
    interaction = (d - c) - (b - a)
    print(f'\nFactorial Analysis:')
    print(f'  ECA effect (vs SE):     {eca_effect:+.1f}% mAP@50')
    print(f'  ReLU6 effect (vs SiLU): {relu6_effect:+.1f}% mAP@50')
    print(f'  Interaction effect:     {interaction:+.1f}% mAP@50')
    print(f'  Total gap explained:    {eca_effect + relu6_effect + interaction:+.1f}%')
    print(f'  Observed gap (D - A):   {d - a:+.1f}%')

    from scipy import stats
    t_stat, p_val = stats.ttest_ind(results['D (ReLU6+ECA)'], results['A (SiLU+SE)'])
    print(f'\n  T-test (D vs A): t={t_stat:.2f}, p={p_val:.4f}')
    sig = 'YES' if p_val < 0.05 else 'NO'
    print(f'  Statistically significant at p<0.05: {sig}')

In [ ]:
output = {
    'experiment': 'cross_ablation_activation_attention',
    'dataset': 'VOC 2007+2012', 'epochs': 300, 'imgsz': 416,
    'seeds': [42, 123, 256],
    'results': {k: {'maps': v, 'mean': float(np.mean(v)), 'std': float(np.std(v))}
                for k, v in results.items() if v},
}
out_path = Path('experiments/results/cross_ablation_summary.json')
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)
print(f'Saved to {out_path}')